#### Import des librairies

In [1]:
import pandas as pd
from tqdm import tqdm
import numpy as np

In [2]:
path_file = '~/Bureau/exports/20241112/AGS_20241112_exports_agronomes_-archive/AGS_20241112_exports_agronomes_assolees_synthetisees.csv'

ENTREPOT_PATH = '~/Bureau/utils/data/'
DIRODUR_FILES_PATH = '~/Bureau/projets/DIRODUR/magasin/'
df = {}

#### Import des données

In [3]:
# -------------------------- #
# IMPORT DES DONNÉES DIRODUR #
# -------------------------- #
dirodur_files = ['correspondance_destination_gcpe']

for file in dirodur_files:
    df[file] = pd.read_csv(DIRODUR_FILES_PATH + file+'.csv', sep = ';')

In [4]:
# ----------------------------------------- #
# CRÉATION DU RÉFÉRENTIEL DE MATCH D'UNITÉS #
# ----------------------------------------- #
dict_unites = {
    'q/ha (humidité ramenée à la norme)' : 'Q_HA_TO_STANDARD_HUMIDITY',
    't MS/ha' : 'TONNE_MS_HA',
    't sucre/ha' : 'TONNE_SUGAR_HA',
    't/ha' : 'TONNE_HA',
    'tonne_racines_ha_16_pourc' : 'TONNE_RACINES_HA_16_POURC', 
    'q/ha' : 'Q_HA',
    'kg/m²' : 'KG_M2',
    'unité/ha' : 'UNITE_HA',
    'hl/ha' : 'HL_HA'
}
df['unite_rendement'] = pd.DataFrame.from_dict(dict_unites, orient='index', columns=['unite_agrosyst']).reset_index().rename(columns={'index':'unite_nl'})

In [5]:
# complétion du référentiel transmis par les agronomes
left = df['correspondance_destination_gcpe']
right = df['unite_rendement']
df['correspondance_destination_gcpe'] = pd.merge(left, right, left_on = 'Unité_rendement', right_on = 'unite_nl', how = 'left')

In [8]:
# ----------------------------- #
# IMPORT DES DONNÉES DATAGROSYST#
# ----------------------------- #
df = {}

def import_df(df_name, path_data, sep, index_col=None):
    df[df_name] = pd.read_csv(path_data+df_name+'.csv', sep = sep, index_col=index_col, low_memory=False).replace({'\r\n': '\n'}, regex=True)

def import_dfs(df_names, path_data, sep = ',', index_col=None, verbose=False):
    for df_name in tqdm(df_names) : 
        if(verbose) :
            print(" - ", df_name)
        import_df(df_name, path_data, sep, index_col=index_col)

tables_with_id = [
    "levier", "modele_decisionnel_strategie"
]

tables_without_id = [
]

# import des données de l'entrepôt avec la colonne 'id' en index 
import_dfs(tables_with_id, ENTREPOT_PATH, sep = ',', index_col='id', verbose=False)

# import des données du magasin
import_dfs(tables_without_id, ENTREPOT_PATH, sep = ',', verbose=False)


100%|██████████| 2/2 [00:00<00:00,  3.29it/s]
0it [00:00, ?it/s]


In [10]:
left = df['modele_decisionnel_strategie']
right = df['levier']
merge = pd.merge(left, right, left_on ='levier_id', right_index=True, how='left')

In [19]:
merge

,levier_id,explication,modele_decisionnel_maitrise_id,code,libelle,type_section,type_strategie
id,,,,,,,
fr.inra.agrosyst.api.entities.managementmode.Strategy_c2701be4-9231-4534-8c3d-a810aa67f822,fr.inra.agrosyst.api.entities.referential.RefS...,Observe ses vignes,fr.inra.agrosyst.api.entities.managementmode.S...,VITI_MALA_LUTT_00475,Observations / comptages,MALADIES,LUTTE_CHIMIQUE
fr.inra.agrosyst.api.entities.managementmode.Strategy_cf275597-11fa-4b4f-b295-f435510bf67c,fr.inra.agrosyst.api.entities.referential.RefS...,lachers d'auxiliares,fr.inra.agrosyst.api.entities.managementmode.S...,MARA_RAVA_BIOC_00662,Autre,RAVAGEURS,PLANTES_SERVICES
fr.inra.agrosyst.api.entities.managementmode.Strategy_0adfe6af-e5f6-485b-aadd-fa2e2a0a29f6,fr.inra.agrosyst.api.entities.referential.RefS...,Observe ses vignes,fr.inra.agrosyst.api.entities.managementmode.S...,VITI_RAVA_LUTT_00504,"Observations, comptages, piégeage",RAVAGEURS,LUTTE_CHIMIQUE
fr.inra.agrosyst.api.entities.managementmode.Strategy_afa764a9-62f8-46e0-96eb-8c66d2f22079,fr.inra.agrosyst.api.entities.referential.RefS...,Arrachage des pieds contaminés,fr.inra.agrosyst.api.entities.managementmode.S...,VITI_RAVA_ACTI_00490,Élimination inoculum (arrachage ceps FD),RAVAGEURS,ACTION_INOCULUM
fr.inra.agrosyst.api.entities.managementmode.Strategy_b23ab788-a84f-473d-aa7c-dfa08c541df9,fr.inra.agrosyst.api.entities.referential.RefS...,"pas d'attaque, jamais de traitement",fr.inra.agrosyst.api.entities.managementmode.S...,POLY_RAVA_LUTT_00407,Raisonnement,RAVAGEURS,LUTTE_CHIMIQUE
...,...,...,...,...,...,...,...
fr.inra.agrosyst.api.entities.managementmode.Strategy_821d5a94-e9de-4ca1-bce6-321566b467cd,fr.inra.agrosyst.api.entities.referential.RefS...,Piège dans une parcelle,fr.inra.agrosyst.api.entities.managementmode.S...,VITI_RAVA_LUTT_00504,"Observations, comptages, piégeage",RAVAGEURS,LUTTE_CHIMIQUE
fr.inra.agrosyst.api.entities.managementmode.Strategy_3badf811-e94f-40fc-bb8c-4e3501d12ecd,fr.inra.agrosyst.api.entities.referential.RefS...,lutte à base de soufre et poudrage si necessaire,fr.inra.agrosyst.api.entities.managementmode.S...,VITI_MALA_LUTT_00476,"Pulvérisateur face/face, traitement ciblées su...",MALADIES,LUTTE_CHIMIQUE
fr.inra.agrosyst.api.entities.managementmode.Strategy_4d23d426-5750-4a98-98e6-1a7254be1066,fr.inra.agrosyst.api.entities.referential.RefS...,Ovicides et Larvicides sur G1 G2 et G3,fr.inra.agrosyst.api.entities.managementmode.S...,VITI_RAVA_LUTT_00504,"Observations, comptages, piégeage",RAVAGEURS,LUTTE_CHIMIQUE


In [18]:
merge.loc[
    (merge['libelle'] == 'Autre') &  (~merge['explication'].isna())
]

,levier_id,explication,modele_decisionnel_maitrise_id,code,libelle,type_section,type_strategie
id,,,,,,,
fr.inra.agrosyst.api.entities.managementmode.Strategy_cf275597-11fa-4b4f-b295-f435510bf67c,fr.inra.agrosyst.api.entities.referential.RefS...,lachers d'auxiliares,fr.inra.agrosyst.api.entities.managementmode.S...,MARA_RAVA_BIOC_00662,Autre,RAVAGEURS,PLANTES_SERVICES
fr.inra.agrosyst.api.entities.managementmode.Strategy_c93edfc6-40bc-4172-9ad6-93b41b8474ba,fr.inra.agrosyst.api.entities.referential.RefS...,"Mise en place d'une bâche tissée sur 5 ha, auc...",fr.inra.agrosyst.api.entities.managementmode.S...,ARBO_ADVE_LUTT_00010,Autre,ADVENTICES,LUTTE_PHYSIQUE
fr.inra.agrosyst.api.entities.managementmode.Strategy_b9bbef1b-cabb-4373-a1b3-c15e2d520089,fr.inra.agrosyst.api.entities.referential.RefS...,maintient des pots de culture grâce à un outil...,fr.inra.agrosyst.api.entities.managementmode.S...,HORTI_RAVA_ATTE_00451,Autre,RAVAGEURS,ATTENUATION_PRESSION_BIOAGRESSEUR
fr.inra.agrosyst.api.entities.managementmode.Strategy_b1a5cd19-8430-49b1-8c87-528445d75811,fr.inra.agrosyst.api.entities.referential.RefS...,vide sanitaire,fr.inra.agrosyst.api.entities.managementmode.S...,HORTI_RAVA_ACTI_00450,Autre,RAVAGEURS,LIMITATION_INOCULUM
fr.inra.agrosyst.api.entities.managementmode.Strategy_533e05dd-3e56-4c5a-be7a-fc113086fe23,fr.inra.agrosyst.api.entities.referential.RefS...,solarisation,fr.inra.agrosyst.api.entities.managementmode.S...,HORTI_RAVA_ACTI_00450,Autre,RAVAGEURS,LIMITATION_INOCULUM
...,...,...,...,...,...,...,...
fr.inra.agrosyst.api.entities.managementmode.Strategy_39ea0272-ff6f-484d-a91d-363e2ecc9467,fr.inra.agrosyst.api.entities.referential.RefS...,Bandes fleuries pour favoriser les auxiliaires.,fr.inra.agrosyst.api.entities.managementmode.S...,MARA_RAVA_LUTT_00665,Autre,RAVAGEURS,LUTTE_BIOLOGIQUE_MACRO_ORGANISMES
fr.inra.agrosyst.api.entities.managementmode.Strategy_d211cf12-a868-4d69-893e-e003c2ed30e7,fr.inra.agrosyst.api.entities.referential.RefS...,Effeuillage.,fr.inra.agrosyst.api.entities.managementmode.S...,MARA_MALA_ACTI_00652,Autre,MALADIES,LIMITATION_INOCULUM
fr.inra.agrosyst.api.entities.managementmode.Strategy_7f8fd299-6292-4484-923e-633684835bfe,fr.inra.agrosyst.api.entities.referential.RefS...,Utilisation de Limocide (terpène d'orange),fr.inra.agrosyst.api.entities.managementmode.S...,ARBO_RAVA_LUTT_00747,Autre,RAVAGEURS,LUTTE_PHYSIQUE
